# Social Physics

**Domain:** Symbolic AI & Logic  ·  **from study list**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _conceptual_

> A self-contained refresher on **social physics**: modeling human social interaction as a rule-driven, simulable system — the way a game engine simulates rigid-body physics, but the "bodies" are people and the "forces" are relationships, status, and social norms. This is the **symbolic-AI / computational-social-simulation** sense (Comme il Faut → Ensemble → Versu), not Alex Pentland's data-science book of the same name (see §1).

## 1. What & Why

**What it is.** Social physics treats social life as a *system you can simulate from rules*. Characters carry **social state** (traits, relationships, status, social networks); they take **social actions** ("flirt", "brag", "ask out", "betray"); and a rule engine computes the outcome — who wants to do what, whether it's accepted, and how everyone's relationships change as a result. The label "physics" is the whole point: just as a physics engine turns *forces → new positions*, a social-physics engine turns *social moves → new social state*, deterministically and generatively, without an author scripting each branch.

**The problem it solves.** Authoring believable, *reactive* characters by hand doesn't scale. Dialogue trees and quest scripts explode combinatorially and break the moment the player does something unanticipated. Social physics replaces "write every branch" with "write the rules of the social world and let drama *emerge*." It is the engine behind **emergent narrative**: the designers of *Prom Week* (2012) authored ~5,000 social rules and got a near-infinite space of student-drama playthroughs.

**Lineage (the canon).**
- **Comme il Faut (CiF)** — UC Santa Cruz Expressive Intelligence Studio; powered *Prom Week*. Coined "social physics" in this sense.
- **Ensemble** — the open-source, JavaScript reimplementation/successor of CiF (see the sibling [`ensemble.ipynb`](ensemble.ipynb)).
- **Versu** — Richard Evans & Emily Short; agents coordinated by *social practices* (reactive joint plans) with utility-based action selection.
- Cousins: *Talk of the Town* / *Bad News*, *The Sims*' relationship model, social-practice and "storylet" systems.

**Reach for it when:** you want NPCs whose relationships and reactions are systemic and emergent — social sims, dating/political sims, interactive drama, relationship-heavy RPGs, drama managers.

**Skip it when:** you want tight authorial control over exact beats (linear/branching authored story), social state is purely cosmetic, or you only need individual task behavior (use a behavior tree / GOAP instead). Social physics buys *emergence* at the cost of *control*.

**Disambiguation.** Alex Pentland's *Social Physics: How Good Ideas Spread* (MIT, 2014) is a **different field** — an empirical, data-driven science of *idea flow* in human networks via "reality mining" (phone/sensor data). Useful and real, but it's network/data science, not symbolic AI. This notebook is about the rule-based simulation sense that lives in Symbolic AI & Logic.

## 2. Mental Model

**It's a physics engine for relationships.**

```
   RIGID-BODY PHYSICS                 SOCIAL PHYSICS
   -----------------                  --------------
   bodies (mass, position)     <->    characters (traits, status)
   forces (gravity, springs)   <->    influence rules (social norms)
   collision                    <->    a social exchange ("ask out")
   integrator (sum forces)     <->    volition (sum influence rules)
   new positions/velocities    <->    new social state (relationships)
```

**The core loop** — accumulate "forces", then integrate:

```
            ┌────────────────────────────────────────────┐
            │  SOCIAL STATE                                │
            │  traits · relationships · networks · status  │
            └───────────────┬──────────────────────────────┘
                            │ (query predicates)
                            ▼
   for each candidate exchange between A and B:
        volition(A→B) = Σ influence_rules matching the social state
                            │
                            ▼
   pick the exchange with highest mutual volition  ──►  perform it
                            │
                            ▼
   apply EFFECTS  ──►  social state changes  ──►  may trigger reactions
                            │
                            └──────────── loop ────────────┘
```

Each character's **volition** for an exchange is a *sum of weighted rules*, exactly like accumulating forces before an integration step. High net volition = the move "wants" to happen. The engine is otherwise just bookkeeping over a database of social facts — close kin to Datalog/Prolog queries (see [`datalog.ipynb`](datalog.ipynb)) with utility scoring bolted on.

## 3. Key Concepts

| Term | What it means |
| --- | --- |
| **Social state** | The full database of facts about the cast: traits, status, and **directed relationships/networks** (friendship, romance, enmity), each a boolean or numeric value from one character toward another. |
| **Character / cast** | The agents. Social state is mostly *relational* — defined *between* characters, not just within one. |
| **Social exchange** (a.k.a. social move / microtheory) | A parameterized interaction — `flirt`, `brag`, `askOut`, `confront` — with an *intent*, *influence rules*, and *effects*. The atomic unit an author writes. |
| **Influence rule** | A weighted rule: *if* these predicates hold in the social state, *then* add ±w to a character's volition for an exchange. This is where authored **social norms** live ("people flirt with those they find attractive: +3"). |
| **Volition** | The computed *desire* of initiator and responder to do/accept an exchange — the sum of matching influence rules. Drives both **selection** (what happens) and **outcome** (accept vs reject). It is *relative*, not absolute. |
| **Trigger rules / Volition rules** | *Trigger rules* keep state consistent by deriving facts ("if A dating B then A and B are not single"). *Volition rules* compute desires. |
| **Predicates / conditions** | Boolean queries over social state — the Prolog/Datalog-like building blocks of rules. |
| **Social network** | A typed, weighted, directed graph (buddy network, romance network, cool network) that exchanges read and modify. |
| **Emergent narrative / drama management** | The payoff: stories nobody scripted, arising from rule interactions. A *drama manager* may nudge selection toward authored arcs. |

## 4. Setup

**This is not Python-runnable.** Social-physics engines are authored data + their own runtime, not a `pip` library you drive from a notebook cell. So below are real CLI commands and config/code *snippets* — there is no `print()` theater here.

The canonical open-source engine is **Ensemble** (JavaScript/Node; the CiF successor). The original CiF was C#; Versu was a proprietary system (Praxis/Prompter, an extended-Datalog stack). To get hands-on:

```bash
# Ensemble — the open-source social-physics engine
git clone https://github.com/ensemble-engine/ensemble.git
cd ensemble

# It's a JS library: include ensemble.js in a web project, or run the
# bundled examples / authoring tool. A minimal Node smoke test:
node -e "const ensemble = require('./js/ensemble/ensemble.js'); console.log(typeof ensemble.init);"
```

Authoring is data-first: you write JSON **schema** (the social-state vocabulary) and JSON/CiF **rules** (exchanges, influence rules, triggers), then load them into the engine and step the simulation. The **Ensemble Authoring Tool** (a desktop app) exists to edit and *debug* those rule sets — debugging emergent rule interactions by hand is the hard part (see §6). See the sibling [`ensemble.ipynb`](ensemble.ipynb) for an engine-specific walkthrough.

## 5. Worked Examples

Conceptual walkthroughs — Ensemble-flavored JSON/pseudo-code, *not executed*.

### Example 1 — Define the social state (schema)

The schema is the vocabulary of your social world: what kinds of facts can exist. Here: a trait, a directed numeric *romance* network, and a boolean *dating* relationship.

```jsonc
// schema.json — the "types" of social state
[
  { "category": "trait",
    "isBoolean": true,  "directionType": "undirected",
    "types": ["shy", "confident", "popular"] },

  { "category": "network",
    "isBoolean": false, "directionType": "directed",
    "minValue": 0, "maxValue": 100,
    "types": ["romance"] },          // how much X is into Y (0–100)

  { "category": "relationship",
    "isBoolean": true,  "directionType": "undirected",
    "types": ["dating", "friends", "enemies"] }
]
```

Initial facts (the starting world state):

```jsonc
// cast: Alice, Bob, Carol
{ "category": "trait",   "type": "shy",     "first": "Alice", "value": true }
{ "category": "network", "type": "romance", "first": "Alice", "second": "Bob", "value": 70 }
{ "category": "network", "type": "romance", "first": "Bob",   "second": "Alice", "value": 20 }
```

### Example 2 — Define an exchange and how *volition* is computed

`askOut` is a social exchange. Its **influence rules** encode social norms as weighted conditions; the engine sums the matching ones to get each character's volition.

```jsonc
// askOut.json
{
  "name": "askOut",
  "intent": { "category": "relationship", "type": "dating", "value": true },
  "influenceRules": [
    { "weight":  5, "conditions": [   // I'm very into them  -> I want this
        {"category":"network","type":"romance","first":"initiator","second":"responder","operator":">","value":60} ] },
    { "weight": -4, "conditions": [   // ...but I'm shy       -> I hold back
        {"category":"trait","type":"shy","first":"initiator","value":true} ] },
    { "weight":  3, "conditions": [   // they're confident    -> easier to approach
        {"category":"trait","type":"confident","first":"responder","value":true} ] }
  ],
  "effects": [ // applied if accepted
    {"category":"relationship","type":"dating","first":"initiator","second":"responder","value":true} ]
}
```

The engine's selection loop, in pseudo-code:

```text
function step(state):
    best = null
    for (A, B) in ordered_pairs(cast):
        for exch in exchanges:
            vA = sum(rule.weight for rule in exch.influenceRules
                                 if rule.conditions hold for (initiator=A, responder=B) in state)
            vB = sum(... for responder B evaluating acceptance ...)   # responder's volition
            score = vA + vB
            if score > best.score: best = (A, B, exch, score)
    if best.vA > 0 and best.vB >= 0:        # initiator wants it, responder doesn't refuse
        apply(best.exch.effects, state)     # mutate social state
        runTriggerRules(state)              # restore derived consistency
        return narrate(best)                # -> "Alice asks Bob out. He says yes."
```

For Alice→Bob above: romance 70 (+5) − shy (−4) + Bob confident? (say true, +3) = **+4** → Alice wants it. The same machinery, run every step over every pair and every exchange, is what produces *emergent* drama: nobody scripted "Alice asks Bob out", the rules did.

### Example 3 — Versu's alternative: social practices (contrast)

Versu doesn't centrally score every pair; instead each agent uses **utility-based action selection**, and coordination comes from **social practices** — reactive joint plans that *afford* actions without controlling agents:

```text
practice Conversation(speaker, listener):
    affords greet, smalltalk, askQuestion, takeLeave
    norm: after a question, the listener *should* answer (raises its utility)
    # agents stay autonomous: the practice only nudges utilities, never forces a move
```

Same goal (emergent, replayable social drama), different mechanism: *centralized rule scoring* (CiF/Ensemble) vs *distributed utility agents + shared practices* (Versu).

## 6. Gotchas & Pitfalls

- **Authoring burden is the real cost.** Believable behavior needs *hundreds to thousands* of influence rules. *Prom Week* shipped ~5,000. Budget for content authoring, not just engine integration.
- **Debugging emergence is brutal.** When a character does something weird, the cause is the *interaction* of dozens of weighted rules, not one line. You can't set a breakpoint on "drama." Tooling that shows *why* a volition was high (rule-by-rule contribution) is essential — and still painful.
- **Volition is relative, not absolute.** A move with volition +4 only happens if nothing else scores higher *right now*. Tuning is about *relative* weights across the whole rule set; nudging one weight can silently starve unrelated behaviors.
- **Weight-tuning is finicky and global.** There's no principled scale. Authors converge on magic numbers by playtesting. Small changes ripple system-wide.
- **Combinatorial cost.** Naively scoring every exchange for every ordered pair every step is O(pairs × exchanges × rules). Fine for a classroom (~10 chars); needs pruning/caching for large casts.
- **The narrative paradox / loss of control.** Emergent systems resist authored arcs. You often *also* need a drama manager or storylets to steer toward intended beats — which partly reintroduces the authoring you were trying to avoid.
- **Players read emergence as bugs.** A surprising-but-valid social event ("why did my friend suddenly date my enemy?") often looks like a glitch. You must *surface the reasons* (visible relationship meters, narrated motivations) or players lose trust.
- **Hardcoded norms are cultural.** Influence rules *are* a model of one culture's social norms. They don't transfer across settings (a flirting rule for a high-school sim is wrong for a Regency drama). Re-author per world.
- **State must stay consistent.** Forget a trigger rule and you get contradictions ("dating but also single"). Derived facts need disciplined maintenance.

## 7. When to Use vs Alternatives

| Approach | What it's good at | Trade-off vs social physics |
| --- | --- | --- |
| **Social physics** (CiF / Ensemble) | Systemic, emergent *relationships*; reactive NPCs; replayability | High authoring + debugging cost; weak authorial control |
| **Dialogue trees / Twine / branching** | Total authorial control; hand-crafted beats | No emergence; combinatorial branch explosion; brittle to player creativity |
| **Storylets / quality-based narrative** | Middle ground — authored chunks gated by state | Less emergent than rules; still scales better than trees |
| **Behavior trees / GOAP / utility AI** | *Individual* task behavior (combat, navigation) | Not relational — no model of *between*-character social state. Often *combined*: utility AI ≈ volition, social physics supplies the relational state. |
| **Versu / social practices** | Emergent drama via autonomous agents + shared practices | Distributed utility model instead of central scoring; different authoring style, similar costs |
| **Agent-based modeling** (NetLogo, Mesa) | *Statistical* macro-behavior of crowds; research | Aims at aggregate patterns, not individual *dramatic* meaning |
| **LLM-driven NPCs** ("generative agents") | Open-ended, fluent, low up-front authoring | Expensive, slow, non-deterministic, hard to constrain/explain; social physics is cheap, explainable, controllable. Increasingly *hybridized* (rules for state, LLM for surface text). |

**Rule of thumb:** choose social physics when *relationships and social consequence are the gameplay* and you can invest in authoring. Choose authored/branching when the *exact story* matters more than emergence. Choose LLMs when *fluent open-ended dialogue* matters more than determinism — and consider combining a social-physics state model with an LLM front-end.

## 8. Resources

**Engines & tools**
- **Ensemble Engine — source** (open-source CiF successor, JS): https://github.com/ensemble-engine/ensemble
- **Ensemble project page** (Ben Samuel): http://www.ben-samuel.com/projects/the-ensemble-engine/
- **Expressive Intelligence Studio** (the lab behind CiF / Prom Week): https://expressiveintelligence.github.io/

**Foundational papers**
- *The Ensemble Engine: Next-Generation Social Physics* (Samuel et al., FDG 2015): http://www.ben-samuel.com/wp-content/uploads/2015/09/FDG2015-The-Ensemble-Engine-Next-Generation-Social-Physics.pdf
- *Versu — A Simulationist Storytelling System* (Evans & Short, IEEE T-CIAIG 2014): https://cs.uky.edu/~sgware/reading/papers/evans2014versu.pdf
- *Road to the IGF: Expressive Intelligence Studio's Prom Week* (design retrospective): https://www.gamedeveloper.com/design/road-to-the-igf-expressive-intelligence-studio-s-i-prom-week-i-

**Context & disambiguation**
- Emily Short, *Versu: Conversation Implementation* (designer's blog): https://emshort.blog/2013/02/26/versu-conversation-implementation/
- Alex Pentland, *Social Physics: How Good Ideas Spread* (the **other**, data-science sense — for contrast): https://www.media.mit.edu/projects/social-physics/overview/

**Cross-links in this library:** [`ensemble.ipynb`](ensemble.ipynb) (engine specifics) · [`datalog.ipynb`](datalog.ipynb) / [`swi-prolog.ipynb`](swi-prolog.ipynb) (the predicate/query substrate) · [`insimul-dsl.ipynb`](insimul-dsl.ipynb) (simulation DSLs).